# BioSense TWR — A100 Simulation Run
**Pipeline**: FDTD wall model → Boulic body → Radar echo synthesis → Back-projection (Rust/PyO3) → AI pose inversion → Muon density → Gravimetry fusion → OKS evaluation

**Compute**: A100 40GB GPU · Chunked checkpoint → Google Drive (5 GB budget)


In [ ]:
# ═══ CELL 1: Environment setup ═══
# pyo3 is a Rust crate — it is NOT a pip package and must not be pip-installed.
# maturin handles the Rust-Python bridge; only maturin is needed here.
import subprocess, sys, os

def run(cmd, env=None):
    """Run shell command, raise on failure, stream output."""
    result = subprocess.run(
        cmd, shell=True, check=True,
        executable='/bin/bash',          # needed for `source` to work
        env=env or os.environ.copy(),
        capture_output=False,
    )
    return result

# ── Rust toolchain ─────────────────────────────────────────────────────────
run('curl https://sh.rustup.rs -sSf | sh -s -- -y --default-toolchain stable --profile minimal')

# Persist cargo/rustc on PATH for all subsequent subprocess calls this session
os.environ['PATH'] = os.environ['PATH'] + ':/root/.cargo/bin'

# Verify
run('rustc --version && cargo --version')

# ── Python deps ─────────────────────────────────────────────────────────────
# maturin:     builds the PyO3 Rust extension (.so) and installs it as a Python module
# zstandard:   fast compression for checkpoint chunks
# NOT listed:  pyo3 — it is declared in Cargo.toml, compiled by maturin, not pip
run(
    'pip install -q '
    '"maturin>=1.4,<2.0" '
    'numpy scipy h5py tqdm pydantic '
    'zstandard '
    'google-auth google-auth-oauthlib google-api-python-client gdown'
)

# ── GPU check ────────────────────────────────────────────────────────────────
run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

print('\nEnvironment ready.')


: 

In [ ]:
# ═══ CELL 2: Mount Drive + clone / sync repo ═══
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_PATH   = '/content/biosense_twr'   # adjust to your repo path
DRIVE_ROOT  = '/content/drive/MyDrive/biosense_twr_poc'
DRIVE_FOLDER_ID = ''  # TODO: fill in your Drive folder ID after creating it

os.makedirs(DRIVE_ROOT, exist_ok=True)

# If running from a cloned repo:
# !git clone https://github.com/YOUR_ORG/biosense-twr.git {REPO_PATH}
# For now assume repo already at REPO_PATH

import sys
sys.path.insert(0, f'{REPO_PATH}/poc')
print(f'Repo path: {REPO_PATH}')

In [ ]:
# ═══ CELL 3: Build PyO3 Rust kernels with maturin ═══
import subprocess, os

RUST_DIR = f'{REPO_PATH}/poc'

# Add cargo to PATH for this session
os.environ['PATH'] = os.environ['PATH'] + ':/root/.cargo/bin'

result = subprocess.run(
    'maturin develop --release',
    shell=True, cwd=RUST_DIR,
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Rust build failed')

# Verify import
from biosense_twr import rust_kernels
print('Rust kernels loaded:', dir(rust_kernels))

In [ ]:
# ═══ CELL 4: Config + checkpoint init ═══
import numpy as np
import torch
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Simulation config
N_SCENARIOS_TOTAL = 2000    # total body pose scenarios to generate
CHUNK_SIZE        = 200     # flush to Drive every 200 scenarios
DURATION_S        = 3.0     # seconds of body motion per scenario
RUN_ID            = f'run_{int(time.time())}'

from biosense_twr.checkpoint_manager import CheckpointManager

ckpt = CheckpointManager(
    drive_folder_id=DRIVE_FOLDER_ID,
    run_id=RUN_ID,
    chunk_size=CHUNK_SIZE,
    compress_level=3,
)
ckpt.init_manifest(N_SCENARIOS_TOTAL)
start_from = ckpt.scenarios_already_done()
print(f'Starting from scenario {start_from}/{N_SCENARIOS_TOTAL}')
print(f'Drive budget remaining: {ckpt.budget_remaining_gb():.2f} GB')

In [ ]:
# ═══ CELL 5: Initialise simulation components ═══
from biosense_twr.simulation.wall_model     import WallConfig, MATERIAL_PARAMS
from biosense_twr.simulation.antenna_array  import MIMOArray
from biosense_twr.signal.boulic_model       import BoulicBody, ActivityType
from biosense_twr.signal.radar_echo         import RadarEchoConfig, synthesise_echo
from biosense_twr.muon_gravity.muon_sim     import MuonSimConfig, simulate_muon_tracks
from biosense_twr.muon_gravity.gravity_sim  import (
    GravimeterConfig, simulate_gravimeter_readings, body_segment_masses
)
from biosense_twr.inversion.pose_net        import BioSensePoseNet, PoseLoss, oks_metric

# Array geometry
array = MIMOArray()
tx_pos = array.tx_positions()
rx_pos = array.rx_positions()

# Voxel grid for back-projection
voxels = array.build_voxel_grid(resolution_m=0.10)
print(f'Voxel grid: {voxels.shape[:3]}  ({np.prod(voxels.shape[:3])} voxels)')

# Radar config (bat adaptive — starts in search mode)
radar_cfg = RadarEchoConfig(mode='search', snr_db=15.0)

# Wall configs to randomise over
WALL_MATS = list(MATERIAL_PARAMS.keys())

# Muon + gravimetry configs
muon_cfg    = MuonSimConfig(scan_duration_min=0.1)  # short scan per scenario
gravity_cfg = GravimeterConfig()

# Pose network
input_shape = voxels.shape[:3]
model = BioSensePoseNet(input_shape=input_shape).to(DEVICE)
model = torch.compile(model)  # torch.compile for A100 speedup
criterion = PoseLoss(alpha=10.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler    = torch.cuda.amp.GradScaler()  # mixed precision

print('All components initialised.')

In [ ]:
# ═══ CELL 6: Main simulation + training loop ═══
from tqdm.auto import tqdm
import random

ACTIVITIES: list[ActivityType] = ['walk', 'run', 'crouch', 'stand', 'wave_arm']
rng = np.random.default_rng(seed=42 + start_from)

def scenario_to_radar_image(
    body: BoulicBody,
    wall: WallConfig,
    frame_idx: int,
    duration_s: float = DURATION_S,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Full pipeline for one scenario frame:
      1. Generate body poses
      2. Synthesise radar echo
      3. Apply wall correction (Rust)
      4. Back-project (Rust)
    Returns: (image (nx,ny,nz), pose_gt (N_JOINTS,3))
    """
    poses = body.generate_poses(duration_s)         # (n_frames, N_JOINTS, 3)
    centres, rcs = body.poses_to_scatterers(poses)  # (n_frames, n_seg, 3)
    n_frames = poses.shape[0]
    fi = min(frame_idx, n_frames - 1)

    atten = 10 ** (-wall.total_attenuation_db(radar_cfg.fc) / 20.0)
    phase = wall.phase_shift_rad(radar_cfg.fc)

    s_raw = synthesise_echo(
        tx_pos, rx_pos,
        centres[fi], rcs,
        atten, phase, radar_cfg,
    )  # (n_tx, n_rx, n_t) complex64

    # Rust wall correction
    s_corr_r, s_corr_i = rust_kernels.apply_wall_correction(
        s_raw.real, s_raw.imag,
        wall.eps_r, wall.thickness_m, radar_cfg.fc, wall.attn_db_per_cm
    )

    # Rust back-projection
    image = rust_kernels.backproject_3d(
        s_corr_r, s_corr_i,
        tx_pos, rx_pos, voxels,
        float(radar_cfg.fs), float(radar_cfg.c0)
    )  # (nx, ny, nz) f32

    pose_gt = poses[fi].astype(np.float32)  # (N_JOINTS, 3)
    return image, pose_gt


# ── Training batch accumulation ──
BATCH_SIZE = 16
batch_images, batch_poses, batch_confs = [], [], []
train_losses, oks_scores = [], []

# Pre-compute voxel grid bounds once (uintp = usize, required by Rust)
vox_min_arr = np.array([voxels[...,0].min(), voxels[...,1].min(), voxels[...,2].min()], dtype=np.float32)
vox_max_arr = np.array([voxels[...,0].max(), voxels[...,1].max(), voxels[...,2].max()], dtype=np.float32)
vox_n_arr   = np.array(voxels.shape[:3], dtype=np.uintp)

pbar = tqdm(range(start_from, N_SCENARIOS_TOTAL), desc='Simulating')
for scenario_i in pbar:

    # ── Randomise scenario parameters ──
    activity  = random.choice(ACTIVITIES)
    mat       = random.choice(WALL_MATS)
    thickness = float(rng.uniform(0.10, 0.35))
    n_walls   = int(rng.choice([1, 2]))
    range_m   = float(rng.uniform(3.0, 25.0))
    height_m  = float(rng.uniform(1.55, 1.95))
    mass_kg   = float(rng.uniform(55, 100))

    wall = WallConfig(material=mat, thickness_m=thickness, num_walls=n_walls)
    body = BoulicBody(
        height_m=height_m, mass_kg=mass_kg, activity=activity,
        body_position=np.array([range_m, 0.0, 0.0], dtype=np.float32)
    )
    frame_idx = int(rng.integers(0, 100))

    # ── Radar image ──
    image, pose_gt = scenario_to_radar_image(body, wall, frame_idx)

    # ── Muon density (fast, low scan time) ──
    in_p, in_d, out_p, out_d = simulate_muon_tracks(
        muon_cfg,
        body_centre_xyz=pose_gt[0],  # pelvis as body centre
        body_mass_kg=mass_kg,
        wall_thickness_m=thickness,
        rng=rng,
    )
    muon_density = rust_kernels.reconstruct_density_map(
        in_p, in_d, out_p, out_d, vox_min_arr, vox_max_arr, vox_n_arr
    )  # (nx, ny, nz)

    # ── Gravimetry ──
    seg_masses = body_segment_masses(mass_kg)
    sensor_pos, delta_g = simulate_gravimeter_readings(
        gravity_cfg, pose_gt, seg_masses, rng=rng
    )
    gravity_mass_map = rust_kernels.estimate_mass_anomaly(
        sensor_pos, delta_g,
        pose_gt,          # joint positions as mass estimate centres
        lambda_reg=1e-6,
        max_iter=50,
    )  # (N_JOINTS,)

    # ── Fuse radar image + muon density ──
    # Normalise both to [0,1] and weighted sum
    img_norm   = image / (image.max() + 1e-9)
    muon_norm  = muon_density / (muon_density.max() + 1e-9)
    fused_image = 0.7 * img_norm + 0.3 * muon_norm  # UWB dominant, muon corroborates

    # GT confidence: all joints visible in simulation
    conf_gt = np.ones(pose_gt.shape[0], dtype=np.float32)

    # ── Checkpoint buffer ──
    ckpt.add({
        'fused_image':     fused_image.astype(np.float16),  # save space
        'pose_gt':         pose_gt,
        'delta_g':         delta_g,
        'gravity_mass':    gravity_mass_map,
        'scenario_params': np.array([range_m, thickness, n_walls, height_m, mass_kg], dtype=np.float32),
    })

    # ── Batch training ──
    batch_images.append(torch.tensor(fused_image[None], dtype=torch.float32))
    batch_poses.append(torch.tensor(pose_gt))
    batch_confs.append(torch.tensor(conf_gt))

    if len(batch_images) == BATCH_SIZE:
        model.train()
        x  = torch.stack(batch_images).unsqueeze(1).to(DEVICE)  # (B,1,nx,ny,nz)
        yp = torch.stack(batch_poses).to(DEVICE)
        yc = torch.stack(batch_confs).to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model(x)
            loss = criterion(pred, yc, yp)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            oks = oks_metric(pred.coords, yp, yc).item()

        train_losses.append(loss.item())
        oks_scores.append(oks)
        ckpt.update_best_oks(oks)

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'OKS': f'{oks:.4f}'})
        batch_images.clear(); batch_poses.clear(); batch_confs.clear()

# Final flush
ckpt.flush(force=True)
print(f'\nSimulation complete. Best OKS: {max(oks_scores, default=0):.4f}')

In [ ]:
# ═══ CELL 7: Save model to Drive ═══
import torch
from pathlib import Path

model_path = Path(DRIVE_ROOT) / 'model_latest.pt'
torch.save({
    'model_state':     model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'best_oks':        max(oks_scores, default=0),
    'train_losses':    train_losses,
    'oks_scores':      oks_scores,
    'run_id':          RUN_ID,
}, str(model_path))

print(f'Model saved → {model_path} ({model_path.stat().st_size / 1024**2:.1f} MB)')

In [ ]:
# ═══ CELL 8: Evaluation + OKS summary plot ═══
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, alpha=0.7)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Batch'); axes[0].set_ylabel('Loss')

axes[1].plot(oks_scores, color='green', alpha=0.7)
axes[1].axhline(0.5, color='red', linestyle='--', label='OKS=0.5 threshold')
axes[1].set_title('OKS Score (higher = better pose match)')
axes[1].set_xlabel('Batch'); axes[1].set_ylabel('OKS')
axes[1].legend()

plt.tight_layout()
fig.savefig(Path(DRIVE_ROOT) / 'training_curves.png', dpi=150)
plt.show()

print(f'Budget used: {(4.5 - ckpt.budget_remaining_gb()):.2f} / 4.5 GB')
print(f'Final OKS (mean last 50): {np.mean(oks_scores[-50:]):.4f}')